# Item 15e — Pesos de atenção do `A` e do `A-emb` (descritivo)

Quinto notebook da Phase 7 ([spec](../.specs/features/phase-7-atencao/spec.md), ATT-06, P2).
Entrada: os checkpoints de `item15b_treinos_atencao_colab.ipynb` (`atencao` e `atencao_emb`).
Saída: a entropia normalizada da atenção por nó de destino em função do grau de entrada,
`entropia_por_grau_*.parquet`, as figuras `.png` e, **se o estrato por grau existir**, o cruzamento
do ganho do `A` sobre o SAGE com o tercil de grau.

> **Descritiva: atenção não é explicação.** Estes pesos descrevem como a atenção se distribui na
> cotrajetória ([ADR-0006](../docs/adr/0006-precompromisso-fase-atencao.md); Jain & Wallace, 2019).
> Não provam que a previsão usa a vizinhança. **A medida causal é a ablação por tipo de aresta**
> (ATT-07), e é ela que decide. Toda figura abaixo leva essa legenda.

**O que se mede.** Para cada nó de destino `i` com grau de entrada `d ≥ 2`, a entropia de
`softmax` sobre seus vizinhos, dividida por `log d`: 1 é atenção uniforme (equivale à média do
SAGE), 0 é atenção toda num vizinho. **Grau 1 vale 0 por convenção** (`log 1` não normaliza e o
`softmax` de um vizinho é fixo), e por isso tem faixa própria. Arestas paralelas (par nos dois
charts) contam como vizinhos separados. A entropia é por cabeça e depois média das cabeças.

**On-chart.** Só entram os nós de destino que estão no chart na semana `w` (`onchart_weeks`, a
mesma definição do item 09), com todas as suas arestas de entrada do snapshot `mask_until(g, w)` e
o grafo completo (leitura principal, sem teto de arestas).

**Mesmos nós e mesmas semanas.** Para cada (regime, seed), o `A` e o `A-emb` são avaliados nas
mesmas semanas e com o mesmo conjunto de nós de destino (a máscara é construída uma vez por semana
e usada pelos dois). O `A-emb` preserva o grau de entrada, então as faixas são comparáveis.

**Runtime: GPU (T4).** O notebook aborta logo depois do setup se não houver GPU (`DEVICE = "cuda"`).
Nada treina aqui: só leitura de checkpoints e um forward por semana. Retomável: cada
(regime, seed) grava seu parquet e o laço pula os prontos.

## 0. Ambiente — Colab (GPU) ou local

No Colab, clona o repositório (código + artefatos de dados versionados) e instala o PyG.
Ative a GPU em *Ambiente de execução → Alterar tipo de runtime → GPU*.
Repo privado: cole um PAT em `GITHUB_TOKEN`.

> Os grafos `hetero_full_current.pt` e `hetero_full_pre_pandemia.pt` precisam estar **commitados**
> antes de clonar: os CSVs brutos da rede de gêneros não são versionados, então o grafo não pode
> ser reconstruído aqui dentro.

In [ ]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN

DATA_FILES = [
    "data/processed/graph/hetero_full_current.pt",
    "data/processed/graph/hetero_full_pre_pandemia.pt",
    "data/processed/graph/node_id_map.json",
    "data/processed/timeseries.parquet",
]

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if IS_COLAB:
    if Path(REPO_DIR, "pyproject.toml").exists():
        # O runtime sobrevive ao restart do kernel: um clone de sessão anterior ficaria
        # para trás em silêncio e o import viria do código velho. Sincroniza sempre.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    else:
        tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
        url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
        print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
        if _clone(url).returncode != 0:
            from getpass import getpass
            tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
            _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()
    os.chdir(REPO_DIR)
    print("commit em uso:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                                           capture_output=True, text=True).stdout.strip())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "pyarrow"], check=True)
    faltando = [f for f in DATA_FILES if not Path(REPO_DIR, f).exists()]
    print("✓ Colab pronto. cwd =", os.getcwd())
    if faltando:
        print("⚠️ FALTAM no repositório:", faltando)
else:
    print("Local — nada a clonar.")

In [ ]:
# Portão de GPU: este notebook treina/avalia modelos e só roda numa GPU do Colab (T4).
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Sem GPU: este notebook precisa de um runtime com GPU (T4). No Colab: Ambiente de "
        "execução → Alterar tipo de runtime → GPU, e rode de novo. Abortado antes de qualquer "
        "treino ou avaliação.")
DEVICE = "cuda"
print("DEVICE =", DEVICE, "|", torch.cuda.get_device_name(0))

In [ ]:
import json
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

_anchor = Path(globals().get("__vsc_ipynb_file__", os.getcwd()))
if not _anchor.exists():
    _anchor = Path(os.getcwd())
ROOT = _anchor if _anchor.is_dir() else _anchor.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"raiz do projeto não encontrada a partir de {_anchor}"
sys.path.insert(0, str(ROOT / "src"))

%matplotlib inline
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from music_diffusion_gnn.evaluation.attention_stats import entropy_by_degree
from music_diffusion_gnn.evaluation.longhits import onchart_weeks
from music_diffusion_gnn.graph.build import graph_path
from music_diffusion_gnn.graph.controls import rewire_preserving_degree
from music_diffusion_gnn.graph.edge_features import causal_edge_features
from music_diffusion_gnn.graph.temporal import mask_until
from music_diffusion_gnn.training.trainer import model_from_checkpoint

GRAPH_DIR = ROOT / "data" / "processed" / "graph"
TS_PATH   = ROOT / "data" / "processed" / "timeseries.parquet"
print("ROOT =", ROOT, "| DEVICE =", DEVICE)

In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/music-influence-gnn")
else:
    BASE = ROOT / "results"
CKPT_DIR = BASE / "atencao" / "treinos"
MATRIZ   = BASE / "avaliacao_matriz"           # item 09: origens e predições do SAGE
AVAL     = BASE / "atencao" / "avaliacao"      # item 15c: predições do A
OUT      = BASE / "atencao" / "pesos"
FIG      = OUT / "figuras"
FIG.mkdir(parents=True, exist_ok=True)
print("checkpoints ←", CKPT_DIR, f"({len(list(CKPT_DIR.glob('gnn_*.pt')))} arquivos)")
print("artefatos   →", OUT)

## 1. Configuração

`N_SEMANAS` semanas por regime, espaçadas por igual entre as semanas de origem do item 09
(`origens_{regime}.parquet`), são as mesmas para todos os modelos. `MIN_NOS` é o mínimo de
(nó, semana) por faixa de grau para a faixa aparecer nas figuras; faixas menores ficam na tabela,
mas não são desenhadas, porque a média de poucos nós não descreve nada.

In [ ]:
MAX_WEEK  = 260
SEEDS     = [42, 43, 44]
REGIMES   = ["current", "pre_pandemia"]
VARIANTES = ["atencao", "atencao_emb"]
N_SEMANAS = 12
MIN_NOS   = 30
REL_PRINCIPAL = "music|cotrajectory|music"     # a relação com grau alto e atributo de aresta

LEGENDA = ("descritiva: atenção não é explicação. A medida causal é a ablação por tipo de "
           "aresta, não estes pesos.")

## 2. Dados, semanas e nós on-chart

As semanas vêm das origens do item 09 e o notebook para se o arquivo não existir. Os nós on-chart
de cada semana são os destinos cuja música está em algum chart nessa semana.

In [ ]:
ts = pd.read_parquet(TS_PATH)
onchart = onchart_weeks(ts, max_week=MAX_WEEK)
print(f"pares (música, chart, semana) on-chart: {len(onchart):,}")

def semanas_amostra(regime: str) -> list[int]:
    p = MATRIZ / f"origens_{regime}.parquet"
    if not p.exists():
        raise RuntimeError(f"{p} não existe. As semanas vêm das origens do item 09 "
                           "(item09_avaliacao_matriz_colab.ipynb).")
    semanas = np.sort(pd.read_parquet(p, columns=["week"])["week"].unique())
    pos = np.unique(np.linspace(0, len(semanas) - 1, N_SEMANAS).round().astype(int))
    return [int(semanas[i]) for i in pos]

def mascaras_onchart(g, semanas: list[int]) -> dict[int, np.ndarray]:
    """{semana: máscara booleana (N_music,)} dos nós de música on-chart nessa semana."""
    idx = {sid: i for i, sid in enumerate(g["music"].song_id)}
    por_semana = defaultdict(set)
    alvo = set(semanas)
    for sid, _chart, wk in onchart:
        if wk in alvo and sid in idx:
            por_semana[wk].add(idx[sid])
    out = {}
    for w in semanas:
        m = np.zeros(g["music"].num_nodes, dtype=bool)
        m[list(por_semana[w])] = True
        out[w] = m
    return out

for regime in REGIMES:
    print(regime, "→ semanas", semanas_amostra(regime))

## 3. Carregar um modelo e o grafo que ele viu

Como no item 15c: `model_from_checkpoint` reconstrói o modelo só com os metadados do checkpoint,
e o `A-emb` usa o mesmo grafo embaralhado do treino (refeito de `graph_seed`). A cabeça temporal
nasce zerada; se continuar zerada, o checkpoint não carregou e o notebook para.

In [ ]:
_cache_g: dict[str, object] = {}

def grafo(regime: str):
    if regime not in _cache_g:
        _cache_g[regime] = torch.load(graph_path(regime, GRAPH_DIR), weights_only=False)
    return _cache_g[regime]

def grafo_da_variante(g, variante: str, graph_seed):
    if variante == "atencao":
        return g
    if variante == "atencao_emb":
        assert graph_seed is not None, "checkpoint atencao_emb sem graph_seed: topologia irrecuperável"
        return rewire_preserving_degree(g, seed=int(graph_seed))
    raise ValueError(variante)

def carregar(variante: str, regime: str, seed: int):
    """Devolve (modelo pronto para eval, grafo da variante, checkpoint) ou (None, None, None)."""
    ck_path = CKPT_DIR / f"gnn_{variante}_{regime}_seed{seed}.pt"
    if not ck_path.exists():
        return None, None, None
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    assert (ck["variante"], ck["split_regime"], ck["seed"]) == (variante, regime, seed), ck_path
    g = grafo(regime)
    modelo = model_from_checkpoint(ck, g.metadata())
    assert modelo.encoder_name == "gatv2", f"{ck_path.name}: o encoder não é a atenção"
    assert modelo.pop_bank is not None, f"{ck_path.name}: checkpoint sem pop_bank"
    assert modelo.head.mlp[-1].weight.abs().max().item() > 0, (
        f"{ck_path.name}: cabeça temporal zerada, o checkpoint não foi carregado")
    g_var = grafo_da_variante(g, ck["variante"], ck["graph_seed"])
    return modelo.to(DEVICE).eval(), g_var, ck

_n_ck = len(list(CKPT_DIR.glob("gnn_atencao*_seed*.pt")))
print(f"{_n_ck} checkpoints de atenção encontrados" + ("" if _n_ck else " — rode o item 15b primeiro"))

## 4. Extrair os pesos de atenção

Um forward do encoder por semana, sobre o snapshot `mask_until(g, w)`, com a popularidade da
semana `w` concatenada às features da música e os atributos de aresta causais, exatamente como
`MusicDiffusionGNN.encode_weeks` faz, mas chamando o encoder com `return_attention=True`. Cada
camada devolve, por relação, `(edge_index, alpha)` com `alpha` de forma `(E, cabeças)`. As
relações cujo destino é `music` entram; as demais não chegam à música.

As arestas de várias semanas são empilhadas com os destinos deslocados por `semana × N_music`,
de modo que cada par (nó, semana) é um nó de destino próprio e o grau é o do snapshot daquela
semana.

In [ ]:
@torch.no_grad()
def extrair_semana(modelo, g_var, w: int):
    """[camada][relação] = (edge_index, alpha) em numpy, do encoder no snapshot da semana w."""
    snap = mask_until(g_var, w)
    x = {k: v.to(DEVICE) for k, v in snap.x_dict.items()}
    edge_index = {k: v.to(DEVICE) for k, v in snap.edge_index_dict.items()}
    x["music"] = torch.cat([x["music"], modelo.pop_bank[w].to(DEVICE)], dim=1)
    edge_attr = {k: v.to(DEVICE) for k, v in causal_edge_features(snap, w).items()}
    _, atencao = modelo.encoder(x, edge_index, edge_attr, return_attention=True)
    return [{et: (e.cpu().numpy(), a.cpu().numpy()) for et, (e, a) in camada.items()}
            for camada in atencao]

def nome_relacao(et: tuple) -> str:
    return "|".join(et)

def tabela_regime_seed(regime: str, seed: int):
    """Entropia por faixa de grau do A e do A-emb nas mesmas semanas e nos mesmos nós."""
    mA, gA, _ = carregar("atencao", regime, seed)
    mE, gE, _ = carregar("atencao_emb", regime, seed)
    if mA is None or mE is None:
        return None, None
    g = grafo(regime)
    n = g["music"].num_nodes
    semanas = semanas_amostra(regime)
    mascaras = mascaras_onchart(g, semanas)

    acum = {v: defaultdict(list) for v in VARIANTES}
    cobertura = []
    for i, w in enumerate(semanas):
        m = mascaras[w]                          # a MESMA máscara para os dois modelos
        for variante, modelo, g_var in (("atencao", mA, gA), ("atencao_emb", mE, gE)):
            cobertura.append({"regime": regime, "seed": seed, "variante": variante,
                              "semana": w, "n_nos_onchart": int(m.sum())})
            for camada, por_relacao in enumerate(extrair_semana(modelo, g_var, w)):
                for et, (e, a) in por_relacao.items():
                    if et[2] != "music":
                        continue
                    keep = m[e[1]]
                    e2 = e[:, keep].copy()
                    e2[1] += i * n
                    acum[variante][(camada, nome_relacao(et))].append((e2, a[keep]))

    tabelas = []
    for variante in VARIANTES:
        for (camada, relacao), partes in acum[variante].items():
            ei = np.concatenate([p[0] for p in partes], axis=1)
            al = np.concatenate([p[1] for p in partes], axis=0)
            t = entropy_by_degree(ei, al, n_dst=n * len(semanas))
            tabelas.append(t.assign(regime=regime, seed=seed, variante=variante,
                                    camada=camada, relacao=relacao, n_semanas=len(semanas)))
    del mA, mE, gA, gE
    torch.cuda.empty_cache()
    return pd.concat(tabelas, ignore_index=True), pd.DataFrame(cobertura)

for regime in REGIMES:
    for seed in SEEDS:
        alvo = OUT / f"entropia_por_grau_{regime}_seed{seed}.parquet"
        if alvo.exists():
            print(f"{alvo.name}: pronto — pulando")
            continue
        tab, cob = tabela_regime_seed(regime, seed)
        if tab is None:
            print(f"{regime}/seed{seed}: falta checkpoint de atencao ou atencao_emb — pulando")
            continue
        cob.to_parquet(OUT / f"cobertura_{regime}_seed{seed}.parquet", index=False)
        tab.to_parquet(alvo, index=False)
        print(f"{alvo.name}: {len(tab):,} linhas")

## 5. Conferência: os mesmos nós e as mesmas semanas

Antes de comparar `A` com `A-emb`, o notebook confere, para cada (regime, seed), que as duas
variantes tiveram as mesmas semanas e o mesmo número de nós on-chart em cada uma, e que os dois
modelos têm tabelas para as mesmas relações e camadas. Se algo diferir, para.

In [ ]:
arq_tab = sorted(OUT.glob("entropia_por_grau_*.parquet"))
arq_cob = sorted(OUT.glob("cobertura_*.parquet"))
if not arq_tab:
    raise SystemExit("nenhuma tabela de entropia ainda: rode a célula da seção 4")
tab = pd.concat([pd.read_parquet(f) for f in arq_tab], ignore_index=True)
cob = pd.concat([pd.read_parquet(f) for f in arq_cob], ignore_index=True)

for (regime, seed), grp in cob.groupby(["regime", "seed"]):
    a = grp[grp.variante == "atencao"].set_index("semana")["n_nos_onchart"]
    e = grp[grp.variante == "atencao_emb"].set_index("semana")["n_nos_onchart"]
    assert a.equals(e), f"{regime}/seed{seed}: semanas ou nós on-chart diferem entre A e A-emb"
for (regime, seed), grp in tab.groupby(["regime", "seed"]):
    chaves = {v: set(map(tuple, grp[grp.variante == v][["camada", "relacao", "degree_bin"]].to_numpy()))
              for v in VARIANTES}
    assert chaves["atencao"] == chaves["atencao_emb"], f"{regime}/seed{seed}: tabelas desalinhadas"

resumo_cob = (cob[cob.variante == "atencao"].groupby(["regime", "seed"])
              .agg(semanas=("semana", "size"), nos_onchart_por_semana=("n_nos_onchart", "mean")))
print("A e A-emb: mesmas semanas e mesmos nós on-chart em todos os (regime, seed):")
display(resumo_cob.round(0))
print("relações com destino music:", sorted(tab.relacao.unique()))

## 6. Entropia por grau de entrada

Uma figura por relação e regime, uma coluna por camada do encoder. Linha: média das seeds; banda:
mínimo e máximo entre as seeds. `A` em azul (círculo, linha cheia) e `A-emb` em laranja (quadrado,
linha tracejada curta): a cor e a forma carregam a identidade juntas. A relação principal é a
cotrajetória; as outras relações com destino na música saem na tabela e numa figura própria.

A faixa de grau 1 é 0 por convenção e não diz nada sobre a atenção. A leitura que interessa é
como a entropia muda com o grau a partir de 2, e se o `A` difere do `A-emb` nessa curva.
Isto é uma descrição: não decide O1 a O4 e não substitui a ablação.

In [ ]:
COR = {"atencao": "#2a78d6", "atencao_emb": "#eb6834"}          # slots 1 e 2 da paleta validada
ESTILO = {"atencao": dict(marker="o", ls="-", label="A (grafo real)"),
          "atencao_emb": dict(marker="s", ls=(0, (4, 2)), label="A-emb (grafo embaralhado)")}
SUPERFICIE, TEXTO, TEXTO2, GRADE = "#fcfcfb", "#0b0b0b", "#52514e", "#e7e6e2"
plt.rcParams.update({
    "figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE, "savefig.facecolor": SUPERFICIE,
    "axes.edgecolor": GRADE, "axes.labelcolor": TEXTO2, "xtick.color": TEXTO2, "ytick.color": TEXTO2,
    "text.color": TEXTO, "axes.titlecolor": TEXTO, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "font.size": 10, "legend.frameon": False,
})

def salvar_figura(fig, nome: str):
    """Toda figura de atenção passa por aqui: leva a legenda descritiva e vai para o Drive."""
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    fig.text(0.01, 0.012, LEGENDA, fontsize=9, color=TEXTO2, ha="left", va="bottom", style="italic")
    fig.savefig(FIG / f"{nome}.png", dpi=150)
    plt.show()
    plt.close(fig)

def figura_entropia(tab: pd.DataFrame, relacao: str, regime: str):
    sub = tab[(tab.relacao == relacao) & (tab.regime == regime)]
    if sub.empty:
        return
    camadas = sorted(sub.camada.unique())
    ordem = sub.drop_duplicates("degree_bin").sort_values("degree_min")["degree_bin"].tolist()
    fig, eixos = plt.subplots(1, len(camadas), figsize=(4.4 * len(camadas), 4.0),
                              sharey=True, squeeze=False)
    for ax, camada in zip(eixos[0], camadas):
        for variante, estilo in ESTILO.items():
            s = sub[(sub.camada == camada) & (sub.variante == variante)]
            g = (s.groupby("degree_bin")
                   .agg(media=("entropy_mean", "mean"), lo=("entropy_mean", "min"),
                        hi=("entropy_mean", "max"), n=("n_nodes", "mean"))
                   .reindex(ordem))
            ok = (g["n"] >= MIN_NOS).to_numpy()
            x = np.arange(len(ordem))[ok]
            if not len(x):
                continue
            ax.fill_between(x, g["lo"].to_numpy()[ok], g["hi"].to_numpy()[ok],
                            color=COR[variante], alpha=0.15, linewidth=0)
            ax.plot(x, g["media"].to_numpy()[ok], color=COR[variante], lw=2, ms=6,
                    markeredgecolor=SUPERFICIE, markeredgewidth=1.5, marker=estilo["marker"],
                    ls=estilo["ls"], label=estilo["label"])
            dy = 7 if variante == "atencao" else -13      # rótulo direto no último ponto
            ax.annotate(estilo["label"].split(" ")[0], (x[-1], g["media"].to_numpy()[ok][-1]),
                        textcoords="offset points", xytext=(0, dy), ha="center",
                        fontsize=8.5, color=TEXTO2)
        ax.set_xticks(range(len(ordem)), ordem, rotation=45, ha="right")
        ax.set_ylim(-0.02, 1.05)
        ax.set_title(f"camada {camada + 1}", fontsize=10, loc="left")
        ax.set_xlabel("grau de entrada no snapshot (faixa 1 = 0 por convenção)")
    eixos[0][0].set_ylabel("entropia normalizada da atenção")
    handles, rotulos = eixos[0][0].get_legend_handles_labels()
    fig.legend(handles, rotulos, loc="upper right", ncol=2, bbox_to_anchor=(0.99, 1.0))
    fig.suptitle(f"Entropia da atenção por grau de entrada — {relacao.replace('|', ' · ')}, {regime}",
                 x=0.01, ha="left", fontsize=11)
    salvar_figura(fig, f"entropia_{relacao.replace('|', '_')}_{regime}")

for regime in REGIMES:
    figura_entropia(tab, REL_PRINCIPAL, regime)

In [ ]:
# Visão em tabela de tudo o que a figura mostra (a figura omite faixas com poucos nós).
resumo_tab = (tab[tab.relacao == REL_PRINCIPAL]
              .groupby(["regime", "camada", "degree_bin", "degree_min", "variante"])
              .agg(entropia=("entropy_mean", "mean"), nos_semana=("n_nodes", "mean"))
              .reset_index())
vista = (resumo_tab.pivot_table(index=["regime", "camada", "degree_min", "degree_bin"],
                                columns="variante", values=["entropia", "nos_semana"])
         .round(3).droplevel("degree_min"))
display(vista)
resumo_tab.to_parquet(OUT / "resumo_entropia_cotrajetoria.parquet", index=False)

# As outras relações com destino na música, na camada 1 (a que lê as features observadas).
for relacao in sorted(set(tab.relacao) - {REL_PRINCIPAL}):
    for regime in REGIMES:
        figura_entropia(tab[tab.camada == 0], relacao, regime)

## 7. Cruzamento com o tercil de grau (condicional)

Hipótese pré-declarada e **exploratória** do ADR-0006: o ganho do `A` sobre o SAGE se concentra
nos nós de grau alto de cotrajetória, onde a média dilui. O cruzamento só roda se o arquivo do
estrato por grau da Phase 6b existir (MAP-08). Hoje o MAP-08 está adiado, então o arquivo ainda
não tem nome definido na Phase 6b; este notebook procura `estrato_grau.parquet` em
`results/mapa_estrutura/` (repositório) e em `mapa_estrutura/` (Drive), com as colunas
`song_id`, `chart`, `origin_week` e `tercil_grau` (rótulo ordenável, do menor ao maior grau,
com tercis definidos nos anos de treino). Se o arquivo vier com outro nome ou esquema, ajuste
`CANDIDATOS_GRAU` aqui; o notebook **diz que pulou** quando não acha nada.

O ganho por origem é `(y − ŷ_SAGE)² − (y − ŷ_A)²` on-chart, com as predições médias das seeds
como no item 10 (positivo = o `A` erra menos). A barra é ± 1 erro-padrão por origem, ingênuo
(não agrupa por música), e serve só para descrever. Nada aqui entra no veredito.

In [ ]:
CANDIDATOS_GRAU = [ROOT / "results" / "mapa_estrutura" / "estrato_grau.parquet",
                   BASE / "mapa_estrutura" / "estrato_grau.parquet"]
COLS_GRAU = ["song_id", "chart", "origin_week", "tercil_grau"]
CHAVE_ORIGEM = ["song_id", "chart", "origin_week", "k"]
arquivo_grau = next((p for p in CANDIDATOS_GRAU if p.exists()), None)

if arquivo_grau is None:
    print("CRUZAMENTO PULADO: o estrato por grau de cotrajetória (Phase 6b, MAP-08) não existe. "
          "Procurei em:\n  " + "\n  ".join(map(str, CANDIDATOS_GRAU)))
else:
    grau = pd.read_parquet(arquivo_grau)
    faltam = [c for c in COLS_GRAU if c not in grau.columns]
    assert not faltam, f"{arquivo_grau.name}: colunas faltando {faltam}"
    print("estrato por grau ←", arquivo_grau, f"({len(grau):,} origens)")

In [ ]:
def predicao_media_das_seeds(pasta: Path, prefixo: str, regime: str):
    arqs = [pasta / f"mode2_{prefixo}_{regime}_seed{s}_completo.parquet" for s in SEEDS]
    arqs = [a for a in arqs if a.exists()]
    if not arqs:
        return None
    return pd.concat([pd.read_parquet(a) for a in arqs], ignore_index=True)

def ganho_por_origem(regime: str):
    a = predicao_media_das_seeds(AVAL, "atencao", regime)
    s = predicao_media_das_seeds(MATRIZ, "proposta", regime)
    if a is None or s is None:
        return None
    base = a.groupby(CHAVE_ORIGEM).agg(y_true=("y_true", "first"), onchart=("onchart", "first"),
                                       y_a=("y_pred", "mean"))
    base = base.join(s.groupby(CHAVE_ORIGEM)["y_pred"].mean().rename("y_sage"), how="inner")
    base = base[base.onchart.astype(bool) & base.y_true.notna()].copy()
    base["ganho"] = (base.y_true - base.y_sage) ** 2 - (base.y_true - base.y_a) ** 2
    return base.reset_index()

def figura_ganho(resumo: pd.DataFrame, regime: str):
    charts = sorted(resumo.chart.unique())
    ordem = sorted(resumo.tercil_grau.unique())
    fig, eixos = plt.subplots(1, len(charts), figsize=(4.6 * len(charts), 4.0), sharey=True,
                              squeeze=False)
    cores_k = {1: "#2a78d6", 2: "#eb6834", 4: "#1baf7a"}        # slots 1 a 3 (validam all-pairs)
    marcas_k = {1: "o", 2: "s", 4: "^"}
    for ax, chart in zip(eixos[0], charts):
        for j, k in enumerate((1, 2, 4)):
            s = resumo[(resumo.chart == chart) & (resumo.k == k)].set_index("tercil_grau").reindex(ordem)
            x = np.arange(len(ordem)) + (j - 1) * 0.12
            ax.errorbar(x, s["ganho_medio"], yerr=s["erro_padrao"], color=cores_k[k], lw=2,
                        marker=marcas_k[k], ms=6, markeredgecolor=SUPERFICIE, capsize=0,
                        label=f"k = {k}")
        ax.axhline(0, color=TEXTO2, lw=0.8)
        ax.set_xticks(range(len(ordem)), [str(t) for t in ordem])
        ax.set_title(chart, fontsize=10, loc="left")
        ax.set_xlabel("tercil de grau de cotrajetória (menor → maior)")
    eixos[0][0].set_ylabel("ganho do A sobre o SAGE (erro² menor = positivo)")
    handles, rotulos = eixos[0][0].get_legend_handles_labels()
    fig.legend(handles, rotulos, loc="upper right", ncol=3, bbox_to_anchor=(0.99, 1.0))
    fig.suptitle(f"Ganho do A sobre o SAGE por tercil de grau — {regime} (exploratório)",
                 x=0.01, ha="left", fontsize=11)
    salvar_figura(fig, f"ganho_por_tercil_grau_{regime}")

if arquivo_grau is not None:
    for regime in REGIMES:
        ganho = ganho_por_origem(regime)
        if ganho is None:
            print(f"{regime}: faltam mode2_atencao (item 15c) ou mode2_proposta (item 09) — pulando")
            continue
        cruz = ganho.merge(grau[COLS_GRAU], on=["song_id", "chart", "origin_week"], how="inner")
        print(f"{regime}: {len(cruz):,} de {len(ganho):,} previsões on-chart têm tercil de grau")
        resumo = (cruz.groupby(["tercil_grau", "chart", "k"])["ganho"]
                      .agg(ganho_medio="mean", erro_padrao=lambda x: x.std(ddof=1) / np.sqrt(len(x)),
                           n="size").reset_index())
        resumo.to_parquet(OUT / f"ganho_por_tercil_grau_{regime}.parquet", index=False)
        display(resumo.round(6))
        figura_ganho(resumo, regime)

## 8. O que levar de volta

Do Drive (`atencao/pesos/`):

- `entropia_por_grau_{regime}_seed{seed}.parquet` — entropia por faixa de grau, camada e relação,
  para `A` e `A-emb`;
- `cobertura_{regime}_seed{seed}.parquet` — as semanas e os nós on-chart usados (iguais nos dois);
- `resumo_entropia_cotrajetoria.parquet` — a tabela que sustenta a figura principal;
- `figuras/*.png` — todas com a legenda "descritiva: atenção não é explicação";
- `ganho_por_tercil_grau_{regime}.parquet` — só se o estrato por grau da Phase 6b existia.

Para a dissertação, a figura entra como descrição da atenção na cotrajetória, ao lado da ablação
por tipo de aresta (ATT-07), que é a medida causal.